# 화질 지표 — 정답이 있을 때와 없을 때

같은 결과 영상을 여섯 가지 잣대로 잰다.

| 갈래 | 지표 | 정답 영상 | 좋은 방향 |
|---|---|---|---|
| **FR-IQA** (Full Reference) | PSNR, SSIM, ERGAS | 있어야 한다 | PSNR·SSIM 높게, ERGAS 낮게 |
| **NR-IQA** (No Reference) | NIQE, BRISQUE, PIQE | 필요 없다 | 모두 낮게 |

세 가지를 확인한다.

1. 같은 영상 세트에 여섯 지표를 모두 계산한다
2. **흐리지만 PSNR 높은 결과**와 **선명하지만 PSNR 낮은 결과**를 맞대 본다 — 지각-왜곡 트레이드오프
3. 정답이 없는 인천에서 NR 값을 어떻게 읽고 임계값을 어떻게 정할지 따져본다

결과 영상은 미리 뽑아 저장소에 올려둔 것을 불러온다. **GPU 가 필요 없다.**

## 1. 준비

In [ ]:
import sys, json, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
for f in ('sr_utils.py', 'iqa.py'):
    urllib.request.urlretrieve(f'{BASE}/lib/{f}', f)
    sys.modules.pop(f[:-3], None)
from sr_utils import *
import iqa
import pandas as pd

RES = f'{BASE}/results/comparison'
V = 'q2'          # 내려받은 파일 캐시 폴더. 결과가 바뀌면 이 값을 올린다
META = json.load(open(fetch(f'{RES}/meta.json', f'{V}/meta.json')))

# NIQE·BRISQUE 는 "정상 영상이란 이런 것" 이라는 기준 모델이 있어야 한다.
# 원 논문은 자연 사진(LIVE)으로 만들지만 위성 영상은 통계가 달라, 이 프로젝트의
# IKONOS HR 로 다시 잡은 것을 쓴다. -> 값은 이 데이터 안에서의 상대 비교로만 읽는다.
for f in ('niqe_model.npz', 'brisque_svr.npz'):
    fetch(f'{BASE}/results/iqa/{f}', f'{V}/iqa/{f}')
iqa.load_models(f'{V}/iqa')

VALS = sorted(k for k in META if k.startswith('val'))
TESTS_ = sorted(k for k in META if k.startswith('test'))
ORDER = ['Bicubic', 'SRCNN', 'VDSR', 'EDSR', 'SRGAN', 'ESRGAN', 'SwinIR', 'HAT']


def load(scene, name):
    return imageio.imread(fetch(f'{RES}/{scene}/{name}.png', f'{V}/{scene}/{name}.png'))


print('검증(정답 있음):', ', '.join(VALS))
print('시험(정답 없음):', ', '.join(TESTS_))
print('지표:', ', '.join(iqa.ALL))

## 2. 표본

검증 4장은 정답 HR 이 있어 여섯 지표를 모두 잴 수 있다.
인천 2장은 실제 Sentinel-2 촬영본이라 정답이 없어 NR 세 가지만 잴 수 있다.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(13, 8.6))
for j, v in enumerate(VALS):
    a = ax[0][j] if j < 3 else ax[1][j - 3]
    a.imshow(load(v, 'HR')); a.set_title(f'{v}  (has HR)', fontsize=10)
for j, t in enumerate(TESTS_):
    a = ax[1][j + 1]
    a.imshow(load(t, 'Bicubic')); a.set_title(f'{t}  (no HR)', fontsize=10)
for r in range(2):
    for c in range(3):
        ax[r][c].set_xticks([]); ax[r][c].set_yticks([])
plt.tight_layout(); plt.show()

## 3. 여섯 지표를 같은 세트에 계산

검증 4장에 대해 여덟 가지 복원 결과를 모두 잰다. 한 칸이 4장 평균이다.

In [ ]:
rows = []
for m in ORDER:
    acc = {k: [] for k in iqa.ALL}
    for v in VALS:
        r = iqa.evaluate(load(v, m), load(v, 'HR'))
        for k, x in r.items():
            acc[k].append(x)
    rows.append({'model': m, **{k: float(np.mean(x)) for k, x in acc.items()}})

df = pd.DataFrame(rows).set_index('model')
arrow = {k: ('↑' if iqa.BETTER[k] == 'high' else '↓') for k in iqa.ALL}
df.columns = [f'{c.upper()} {arrow[c]}' for c in df.columns]
print('화살표 방향이 좋은 쪽이다.  ↑ 높을수록 좋다,  ↓ 낮을수록 좋다\n')
print(df.round(3).to_string())

In [ ]:
# 지표마다 1등이 다르다. 각 지표에서 가장 좋은 모델을 뽑아 본다
best = {}
for c, k in zip(df.columns, iqa.ALL):
    best[c] = df[c].idxmax() if iqa.BETTER[k] == 'high' else df[c].idxmin()
for c, m in best.items():
    print(f'{c:12s} 1위  {m}')

## 4. 지각-왜곡 트레이드오프

위 표에서 **FR 1위와 NR 1위가 다른 모델**이다. 두 모델을 직접 맞대 본다.

- FR 이 높은 쪽: 정답 화소값에 가깝게 가려고 애매한 곳을 평균으로 메운다 → 흐려진다
- NR 이 좋은 쪽: 그럴듯한 질감을 만들어 넣는다 → 선명하지만 정답과 화소가 어긋난다

어느 쪽이 "좋은" 결과인지는 지표가 정하지 못한다. 용도가 정한다.

In [ ]:
FR_BEST = df[df.columns[0]].idxmax()        # PSNR 1위
NR_BEST = df[df.columns[3]].idxmin()        # NIQE 1위
print(f'PSNR 1위 = {FR_BEST},  NIQE 1위 = {NR_BEST}\n')

CENTER, SIZE = (67, 370), 70               # 다른 페이지와 같은 자리
for v in VALS[:2]:
    zoom([('Bicubic', load(v, 'Bicubic')),
          (f'{FR_BEST} (best PSNR)', load(v, FR_BEST)),
          (f'{NR_BEST} (best NIQE)', load(v, NR_BEST)),
          ('Target HR', load(v, 'HR'))],
         center=CENTER, size=SIZE, title=v)

In [ ]:
# 가로축 왜곡(PSNR), 세로축 지각(NIQE). 왼쪽 위로 갈수록 둘 다 좋은데, 그런 점이 없다
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, (xc, yc) in zip(ax, [(df.columns[0], df.columns[3]),
                            (df.columns[0], df.columns[4])]):
    a.scatter(df[xc], df[yc], s=90, color='#4f7fa8', zorder=3)
    for m, r in df.iterrows():
        a.annotate(m, (r[xc], r[yc]), fontsize=10,
                   xytext=(6, 4), textcoords='offset points')
    a.set_xlabel(xc + '  (right = better)', fontsize=11)
    a.set_ylabel(yc + '  (down = better)', fontsize=11)
    a.grid(alpha=.3)
    a.set_title('distortion vs perception', fontsize=12)
plt.tight_layout(); plt.show()

### PIQE 는 왜 반대로 나오나

NIQE·BRISQUE 는 선명한 쪽을 좋게 보는데 **PIQE 는 bicubic 을 가장 좋게 본다.**
셋 다 NR 인데 방향이 갈린다.

PIQE 가 재는 것은 "이음매와 잡음이 있는가" 다. 흐릿하기만 한 영상은 이음매도 잡음도
없으니 좋은 점수를 받는다. **선명함을 재는 지표가 아니다.**

NR 지표를 쓸 때 가장 먼저 확인할 것이 이것이다 — 이 지표가 무엇을 벌점으로 삼는가.

In [ ]:
sub = df[[df.columns[0], df.columns[3], df.columns[4], df.columns[5]]]
r = pd.DataFrame({c: (sub[c].rank(ascending=(iqa.BETTER[k] != 'high')).astype(int))
                  for c, k in zip(sub.columns, ['psnr', 'niqe', 'brisque', 'piqe'])})
print('지표별 순위 (1 = 가장 좋음)\n')
print(r.to_string())

## 5. 정답이 없는 실제 운용 — 인천

인천은 실제 Sentinel-2 촬영본이라 정답 HR 이 없다. PSNR·SSIM·ERGAS 를 못 낸다.
남는 것은 NR 세 가지뿐이다.

In [ ]:
rows = []
for m in ORDER:
    acc = {k: [] for k in iqa.NR}
    for t in TESTS_:
        r = iqa.evaluate(load(t, m))
        for k in iqa.NR:
            acc[k].append(r[k])
    rows.append({'model': m, **{k: float(np.mean(x)) for k, x in acc.items()}})
dt = pd.DataFrame(rows).set_index('model')
dt.columns = [f'{c.upper()} ↓' for c in dt.columns]
print('인천 2구역 평균 — 정답이 없어 NR 만 잴 수 있다\n')
print(dt.round(3).to_string())

### 임계값을 어떻게 정할 것인가

절대 기준은 없다. NIQE 3.8 이 "좋다" 는 뜻이 아니다 — 이 값은 기준 모델을 무엇으로
잡았느냐에 따라 통째로 움직인다. 실제로 쓰려면 **기준선을 스스로 만들어야** 한다.

1. **정답을 아는 구간에서 기준선을 뽑는다.** 검증셋처럼 HR 이 있는 데이터에서
   NR 값과 FR 값을 함께 재 두면, "NR 이 이 값이면 PSNR 이 대략 얼마" 라는 대응표가 생긴다
2. **입력 자체의 값을 바닥으로 삼는다.** 복원 결과가 입력(bicubic)보다 NR 이 나쁘면
   무언가 잘못된 것이다. 이건 정답 없이도 판정할 수 있다
3. **분포로 정한다.** 같은 센서·같은 지역의 결과를 여러 장 모아 분포를 보고,
   하위 5% 처럼 상대 위치로 경보를 건다

아래는 1번을 실제로 해 보는 것이다.

In [ ]:
# 검증셋에서 NR 과 FR 을 함께 재어 대응 관계를 만든다 (패치 단위)
pt = []
for m in ORDER:
    for v in VALS:
        sr, hr = load(v, m), load(v, 'HR')
        r = iqa.evaluate(sr, hr)
        pt.append((m, v, r['psnr'], r['niqe'], r['piqe']))
P = pd.DataFrame(pt, columns=['model', 'scene', 'psnr', 'niqe', 'piqe'])

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, k in zip(ax, ['niqe', 'piqe']):
    a.scatter(P[k], P['psnr'], s=45, alpha=.75, color='#4f7fa8')
    c = np.corrcoef(P[k], P['psnr'])[0, 1]
    a.set_xlabel(f'{k.upper()}  (no reference needed)', fontsize=11)
    a.set_ylabel('PSNR  (needs reference)', fontsize=11)
    a.set_title(f'corr {c:+.3f}', fontsize=12); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

print('상관이 뚜렷하면 그 NR 값으로 PSNR 을 대신 가늠할 수 있다.')
print('부호가 반대이거나 상관이 약하면, 그 지표는 이 데이터에서 대리 지표로 쓸 수 없다.')

In [ ]:
# 2번 방법 — 입력(bicubic) 을 바닥으로 삼는 판정
base = dt.loc['Bicubic']
print('인천에서 bicubic 보다 나쁜 항목이 있으면 표시한다\n')
for m in ORDER[1:]:
    bad = [c for c in dt.columns if dt.loc[m, c] > base[c]]
    mark = '주의: ' + ', '.join(c.split()[0] for c in bad) if bad else '이상 없음'
    print(f'{m:9s} {mark}')

## 6. 정리

**지표마다 1등이 다르다.** FR 세 가지는 서로 비슷하게 움직이지만, NR 은 무엇을
벌점으로 삼느냐에 따라 순위가 갈린다. 특히 PIQE 는 흐린 영상을 좋게 본다.

**지각-왜곡 트레이드오프는 실재한다.** GAN 계열은 PSNR 이 bicubic 보다도 낮은데
NIQE·BRISQUE 는 압도적으로 좋다. 화소를 맞히는 일과 그럴듯해 보이는 일은 서로 다른
목표이고, 한쪽을 올리면 다른 쪽이 내려간다.

**NR 값은 절대 기준이 없다.** 기준 모델을 무엇으로 잡았느냐에 따라 값이 통째로
움직이므로, 다른 논문의 수치와 직접 견주면 안 된다. 실제로 쓰려면 정답을 아는
구간에서 대응표를 만들거나, 입력 자체를 바닥으로 삼거나, 분포에서 상대 위치를 본다.

**그래서 무엇을 볼 것인가.** 목적이 지도 제작이나 변화 탐지처럼 화소값이 중요한
일이면 FR 을 본다. 사람이 눈으로 판독하는 일이면 NR 과 실제 확대 그림을 함께 본다.
지표 하나로 결론을 내지 않는 것이 이 페이지의 요지다.